# Construction Helmet Detection with YOLOv11

This notebook trains an Ultralytics **YOLOv11** model on the Construction Helmet Detection dataset () for **100 epochs**.
- **Reference**: Based on 
- **Task**: Object Detection (,  classes)
- **Epochs**: 100
- **Base Model**:  (YOLOv11 Medium)
- **Hardware**: NVIDIA H200 NVL


In [ ]:
# 1. Libraries and Setup
import os
import cv2
import random
import shutil
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import ultralytics
from ultralytics import YOLO

# Disable wandb to prevent login prompt
import wandb
wandb.init(mode="disabled")

print(f"Ultralytics version: {ultralytics.__version__}")

In [ ]:
# 2. Dataset Paths
dataset_dir = "/home/ub/Desktop/COMBINATION/construction_helmet_yolov11"
train_path = os.path.join(dataset_dir, "train")
test_path = os.path.join(dataset_dir, "test")
val_path = os.path.join(dataset_dir, "valid")
data_yaml_path = os.path.join(dataset_dir, "data.yaml")

print(f"Dataset root: {dataset_dir}")
print(f"YAML file: {data_yaml_path}")

In [ ]:
# 3. Check Dataset Counts
train_images = os.listdir(os.path.join(train_path, "images"))
test_images = os.listdir(os.path.join(test_path, "images"))
val_images = os.listdir(os.path.join(val_path, "images"))
train_labels = os.listdir(os.path.join(train_path, "labels"))
test_labels = os.listdir(os.path.join(test_path, "labels"))
val_labels = os.listdir(os.path.join(val_path, "labels"))

print(f"Images -> Train: {len(train_images)}, Valid: {len(val_images)}, Test: {len(test_images)}")
print(f"Labels -> Train: {len(train_labels)}, Valid: {len(val_labels)}, Test: {len(test_labels)}")

In [ ]:
# 4. Helper Function: Plot images with Bounding Boxes
def plot_sample_with_boxes(img_dir, lbl_dir, image_list, num_samples=10, class_names=["helmet", "unhelmet"]):
    colors = {"helmet": "lime", "unhelmet": "red"}
    selected = random.sample(image_list, min(num_samples, len(image_list)))
    cols = 5
    rows = (len(selected) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
    axes = axes.flatten() if rows * cols > 1 else [axes]
    
    for idx, img_name in enumerate(selected):
        img_p = os.path.join(img_dir, img_name)
        lbl_p = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + ".txt")
        img = cv2.imread(img_p)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        ax = axes[idx]
        ax.imshow(img)
        ax.set_title(img_name[:18] + "...", fontsize=9)
        ax.axis("off")
        
        if os.path.exists(lbl_p):
            with open(lbl_p, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id, xc, yc, bw, bh = map(float, parts[:5])
                        cls_name = class_names[int(cls_id)] if int(cls_id) < len(class_names) else f"cls_{int(cls_id)}"
                        box_color = colors.get(cls_name, "yellow")
                        x1 = (xc - bw / 2) * w
                        y1 = (yc - bh / 2) * h
                        box_w = bw * w
                        box_h = bh * h
                        rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor=box_color, facecolor="none")
                        ax.add_patch(rect)
                        ax.text(x1, max(0, y1 - 4), cls_name, color=box_color, fontsize=8, weight="bold")
    
    for ax in axes[len(selected):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# 5. Visualize Sample Training Images
plot_sample_with_boxes(
    os.path.join(train_path, "images"),
    os.path.join(train_path, "labels"),
    train_images,
    num_samples=10
)

In [ ]:
# 6. Initialize YOLOv11 Model (YOLO11 Medium)
model = YOLO("yolo11m.pt")
print(f"Loaded model architecture: {model.model.__class__.__name__}")

In [ ]:
# 7. Train YOLOv11 for 100 Epochs
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    workers=8,
    project="/home/ub/Desktop/COMBINATION/runs/helmet_yolo11",
    name="yolo11m_helmet_100e",
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    verbose=True
)

In [ ]:
# 8. Evaluate Trained Model on Test Split
metrics = model.val(
    data=data_yaml_path,
    split="test",
    project="/home/ub/Desktop/COMBINATION/runs/helmet_yolo11",
    name="test_evaluation",
    exist_ok=True
)
print(f"Test mAP50: {metrics.box.map50:.4f}")
print(f"Test mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# 9. Plot Training Loss and Metrics Curves
results_csv = "/home/ub/Desktop/COMBINATION/runs/helmet_yolo11/yolo11m_helmet_100e/results.csv"
results_png = "/home/ub/Desktop/COMBINATION/runs/helmet_yolo11/yolo11m_helmet_100e/results.png"

if os.path.exists(results_png):
    plt.figure(figsize=(16, 10))
    img = plt.imread(results_png)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Helmet Detection - Training and Validation Metrics", fontsize=14)
    plt.show()
elif os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(df["epoch"], df["train/box_loss"], label="Train Box Loss")
    plt.plot(df["epoch"], df["val/box_loss"], label="Val Box Loss")
    plt.legend()
    plt.title("Box Loss")
    plt.subplot(1, 2, 2)
    plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
    plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
    plt.legend()
    plt.title("mAP Metrics")
    plt.show()

In [ ]:
# 10. Sample Test Predictions
sample_test_images = [os.path.join(test_path, "images", f) for f in os.listdir(os.path.join(test_path, "images"))[:6]]
pred_results = model.predict(sample_test_images, conf=0.35, imgsz=640)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for i, r in enumerate(pred_results):
    im_bgr = r.plot()
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    axes[i].imshow(im_rgb)
    axes[i].axis("off")
    axes[i].set_title(f"Test Sample {i+1} (Detections: {len(r.boxes)})")
plt.tight_layout()
plt.show()